# ASSIGNMENT NLP-5: Fine-Tuning BERT for POS Tagging & Chunking

**Objective:** Build and fine-tune a transformer model (BERT/DistilBERT) to perform Part-of-Speech (POS) Tagging and Chunking (Phrase Detection) using token classification techniques.

---
**Pipeline:**
> Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison

# Fine-Tuning BERT for POS Tagging (Token Classification)

This project focuses on building a token classification model using BERT to perform Part-of-Speech (POS) tagging.

The model is trained on the Universal Dependencies dataset and evaluated using sequence-based metrics.

## Objective

The goal of this assignment is to fine-tune a transformer-based model (BERT) to perform Part-of-Speech (POS) tagging.

This involves:
- Token classification using BERT
- Handling tokenization and label alignment
- Training and evaluating the model
- Performing inference on custom sentences

## Dataset Used: CoNLL-2003

Reason:
Universal Dependencies dataset was not supported due to library updates.
CoNLL-2003 provides both POS tagging and chunking labels, making it suitable for this task.

# Install & Import

In [5]:
!pip install transformers datasets==2.18.0 seqeval


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 7.4 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=a52c09c6af92a6bc69c5d7d52fe407bd328cdc6c94218dfe2ae307219d0f5cfd
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This 

In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
from seqeval.metrics import classification_report

# Load Dataset

In [7]:
dataset = load_dataset("conll2003")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1461: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

# Tokenization + Label Alignment

In [9]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

label_list = dataset["train"].features["pos_tags"].feature.names

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            else:
                label_ids.append(label[word_id])

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

# Model Setup

In [10]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

# Training

In [11]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator
)

In [18]:
trainer.train()

Step,Training Loss
500,0.232658
1000,0.198490
1500,0.165418


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
500,0.232658
1000,0.198490
1500,0.165418


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1756, training_loss=0.19384426951136838, metrics={'train_runtime': 12522.787, 'train_samples_per_second': 2.242, 'train_steps_per_second': 0.14, 'total_flos': 686666534708526.0, 'train_loss': 0.19384426951136838, 'epoch': 2.0})

# Evaluation

In [12]:
predictions, labels, _ = trainer.predict(tokenized_dataset["validation"])

preds = np.argmax(predictions, axis=2)

true_labels = [[label_list[l] for l in label if l != -100] for label in labels]
true_preds = [[label_list[p] for p, l in zip(pred, label) if l != -100]
              for pred, label in zip(preds, labels)]

print(classification_report(true_labels, true_preds))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

              precision    recall  f1-score   support

           '       0.00      0.00      0.00        11
           B       0.03      0.02      0.03      1907
          BD       0.00      0.00      0.00      2224
          BG       0.01      0.02      0.01       699
          BN       0.01      0.00      0.01       928
          BP       0.00      0.00      0.00       365
          BR       0.00      0.04      0.00        53
          BS       0.00      0.00      0.00        18
          BZ       0.00      0.00      0.00       509
           C       0.00      0.00      0.00       932
           D       0.09      0.02      0.04      3231
          DT       0.00      0.02      0.00       162
           H       0.00      0.00      0.00         5
           J       0.01      0.02      0.02      2765
          JR       0.00      0.00      0.00       105
          JS       0.00      0.00      0.00        78
           N       0.08      0.01      0.01      8147
          NP       0.06    

# Inference

In [13]:
sentence = "John works at Google in California"

inputs = tokenizer(sentence, return_tensors="pt")
outputs = model(**inputs).logits

predictions = outputs.argmax(dim=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
predicted_labels = [label_list[p.item()] for p in predictions[0]]

for token, label in zip(tokens, predicted_labels):
    print(token, label)

[CLS] VB
john JJS
works #
at WDT
google VBP
in SYM
california NN|SYM
[SEP] #


## Comparison – POS Tagging vs. Chunking

We now compare the two tasks both conceptually and based on evaluation metrics.

POS Tagging vs Chunking:

POS Tagging:
- Assigns grammatical labels (NOUN, VERB)
- Works at word level
- Easier task

Chunking:
- Groups words into phrases (NP, VP)
- Works at phrase level
- More complex

Conclusion:
Chunking provides deeper syntactic understanding than POS tagging.

### Conceptual Comparison

| Aspect | POS Tagging | Chunking (Phrase Detection) |
|---|---|---|
| **Goal** | Label each word with its grammatical role | Group words into syntactic phrases |
| **Granularity** | Word-level | Phrase-level (multi-word spans) |
| **Example** | "runs" → VBZ | "The quick dog" → B-NP, I-NP, I-NP |
| **Complexity** | Easier – well-defined single label per word | Harder – BIO encoding, span boundaries |
| **Label Space** | Larger (~45 Penn Treebank tags) | Smaller but structured (BIO + phrase types) |
| **Typical F1** | ~97% (high, well-studied task) | ~92-95% (more ambiguous boundaries) |
| **Use Case** | Grammar parsing, downstream NLP | Information extraction, syntax trees |

**Key Insight:**  
POS tagging focuses on the **function** of individual words (noun, verb, adjective), while chunking identifies **structural groups** of words (noun phrase, verb phrase). Both are foundational to higher-level NLP tasks like dependency parsing and information extraction.